# Qwen3 0.6B Causal LM Sweep — Generation + Parsing

This launcher runs the same 16 hyperparameter configs as the next-token sweep, but evaluates the causal LM by generating a short answer and parsing the generated text into `ham`/`spam`.

Use this as the thesis stepping-stone method before the cleaner next-token scorer.

In [ ]:
from pathlib import Path
import os
import signal
import subprocess
import sys

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "dataset").is_dir() and (candidate / "lora-fine-tuning").is_dir():
            return candidate
    raise RuntimeError("Could not find project root.")

PROJECT_ROOT = find_project_root()
METHOD_DIR = PROJECT_ROOT / "lora-fine-tuning" / "methods" / "02_causal_lm_generation_parsing"
SCRIPT = METHOD_DIR / "qwen3_0.6b_generation_parsing_sweep.py"
RESULTS_ROOT = METHOD_DIR / "results"

# Safe default: one tiny smoke run. Flip these for a real A100 run.
RUN_FULL_SWEEP = False
SMOKE_TEST = True
CONFIG_INDEXES = [1]
SWEEP_ID = None

print(f"Project root: {PROJECT_ROOT}")
print(f"Script:       {SCRIPT}")
print(f"Results:      {RESULTS_ROOT}")

In [ ]:
subprocess.run([sys.executable, str(SCRIPT), "list-configs"], cwd=PROJECT_ROOT, check=True)

In [ ]:
def terminate_process_tree(process: subprocess.Popen, timeout: float = 30.0) -> None:
    if process.poll() is not None:
        return
    if os.name == "nt":
        process.terminate()
    else:
        try:
            os.killpg(process.pid, signal.SIGINT)
        except Exception:
            process.send_signal(signal.SIGINT)
    try:
        process.wait(timeout=timeout)
        return
    except subprocess.TimeoutExpired:
        pass
    if os.name == "nt":
        process.kill()
    else:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except Exception:
            process.kill()
    process.wait()

def run_command(command: list[str]) -> None:
    print(" ".join(command))
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, text=True, start_new_session=(os.name != "nt"))
    try:
        return_code = process.wait()
    except KeyboardInterrupt:
        terminate_process_tree(process)
        raise
    if return_code != 0:
        raise RuntimeError(f"Command exited with code {return_code}")

In [ ]:
command = [
    sys.executable,
    str(SCRIPT),
    "run-sweep",
    "--results-root",
    str(RESULTS_ROOT),
    "--resume",
]
if SWEEP_ID:
    command.extend(["--sweep-id", SWEEP_ID])
if not RUN_FULL_SWEEP:
    for index in CONFIG_INDEXES:
        command.extend(["--config-index", str(index)])
if SMOKE_TEST:
    command.extend([
        "--allow-non-cuda",
        "--max-steps", "1",
        "--train-limit", "24",
        "--validation-limit", "8",
        "--test-limit", "8",
    ])
else:
    cuda_check = subprocess.run(
        [sys.executable, "-c", "import torch; raise SystemExit(0 if torch.cuda.is_available() else 1)"],
        cwd=PROJECT_ROOT,
    )
    if cuda_check.returncode != 0:
        raise RuntimeError("CUDA is not available. Keep SMOKE_TEST=True for local checks or run this on the A100 box.")

run_command(command)

In [ ]:
import pandas as pd

sweep_dirs = sorted((RESULTS_ROOT / "sweeps").glob("qwen3_clm_generation_parsing_*"))
if not sweep_dirs:
    raise FileNotFoundError(f"No generation/parsing sweeps found under {RESULTS_ROOT / 'sweeps'}")
latest_sweep_dir = sweep_dirs[-1]
summary_path = latest_sweep_dir / "summary.csv"
print(summary_path)
summary = pd.read_csv(summary_path)
display(summary.sort_values("validation_f1", ascending=False, na_position="last"))